In [2]:
import argparse
import os

# # Disable TorchInductor as requested
# os.environ["TORCHINDUCTOR_DISABLE"] = "1"
# os.environ["TORCH_COMPILE"] = "0"
# os.environ["TORCHDYNAMO_DISABLE"] = "1"
# os.environ["DISABLE_TORCH_COMPILE"] = "1"
# os.environ["TRANSFORMERS_NO_COMPILE"] = "1"

import pandas as pd
import torch
from datasets import load_dataset
from tqdm import tqdm
from transformers import (AutoModelForCausalLM,
                          AutoModelForSequenceClassification, AutoTokenizer)

In [6]:
# !pip install --upgrade transformers
import torch
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig)
model_name = "allenai/OLMo-2-0425-1B-RLVR1"  #"allenai/OLMo-2-1124-7B-DPO" #"allenai/OLMo-2-0425-1B-RLVR1" #"google/gemma-2-2b" #meta-llama/Llama-3.2-3B"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
bnb_config_2 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    revision="step_200",
    torch_dtype=torch.bfloat16,
    # quantization_config = bnb_config_2,
)


# import re
# import numpy as np
# output_dir = "/data/erblina/Master_thesis"
# safe_model_name = re.sub(r'[\\/*?:"<>|]', "_", model_name)
# # os.makedirs(f"{output_dir}/{safe_model_name}", exist_ok=True)
# labels = np.load(f"{output_dir}/{safe_model_name}/labels_new.npy")
# np.save(f"{output_dir}/{safe_model_name}/labels.npy", labels)



In [4]:
decoder_layer = model.model.layers[5]
print(f"Decoder layer: {decoder_layer}")

Decoder layer: Olmo2DecoderLayer(
  (self_attn): Olmo2Attention(
    (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
    (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
    (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
    (q_norm): Olmo2RMSNorm((2048,), eps=1e-06)
    (k_norm): Olmo2RMSNorm((2048,), eps=1e-06)
  )
  (mlp): Olmo2MLP(
    (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
    (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
    (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
    (act_fn): SiLU()
  )
  (post_attention_layernorm): Olmo2RMSNorm((2048,), eps=1e-06)
  (post_feedforward_layernorm): Olmo2RMSNorm((2048,), eps=1e-06)
)


In [19]:
# print(model.model.layers)
for l in model.model.layers:
    continue

print(l.self_attn)

print(model.config.num_attention_heads)

    # print(len(l.self_attn))
   

LlamaAttention(
  (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
  (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
  (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
  (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
)
24


In [ ]:
from torch import nn
def inject_identity_layers(model):
    # this assumes your model has a .transformer.h list of blocks
    for i, block in enumerate(model.transformer.h):
        # attach an identity module
        block.identity = nn.Identity()
        # stash the old forward
        original_forward = block.forward
        # define a new forward that applies the identity to the post‐residual output
        def patched_forward(self, x, **kwargs):
            # run the original block (attention + MLP + residual adds inside)
            out = original_forward(x, **kwargs)
            # now pass that result through the identity
            return self.identity(out)
        # bind it
        block.forward = patched_forward.__get__(block, block.__class__)

# inject_identity_layers(model)